# LLM Experimentation - Quick Test

This notebook allows you to test the 3 LLMs on a subset of test cases to quickly verify everything works.

**Models:**
1. Claude Sonnet 4.5 (Anthropic)
2. GPT 4.1 (OpenAI)
3. GPT 4o Mini (OpenAI)

**Test Cases:** 5 simple queries
**Expected Cost:** ~$0.06-0.10
**Expected Time:** 2-3 minutes

## 1. Setup and Imports

In [25]:
import os
import sys
from pathlib import Path
import json
from datetime import datetime

# Add parent directory to path
parent_dir = Path.cwd().parent
sys.path.insert(0, str(parent_dir))

from llm_evaluator import LLMEvaluator
from test_cases import TEST_CASES

print("✓ Imports successful")
print(f"Working directory: {parent_dir}")

✓ Imports successful
Working directory: c:\Users\Vamsi Chintalapati\Desktop\DSBA 6010\milestone_3_llm_experiments


In [26]:
# Force reload the module to get latest changes
import importlib
import llm_evaluator
importlib.reload(llm_evaluator)
from llm_evaluator import LLMEvaluator

print("✓ Module reloaded with latest changes")

✓ Module reloaded with latest changes


## 2. Initialize Evaluator

In [27]:
# Create evaluator instance
evaluator = LLMEvaluator()

print("\n📊 Configured Models:")
for model_key, config in evaluator.models.items():
    print(f"  - {config['name']} ({config['provider']})")
    print(f"    Model ID: {config['model_id']}")
    print(f"    Cost: ${config['input_cost']}/M input, ${config['output_cost']}/M output")
    print()

✓ Connected to AdventureWorksDW2025
✓ Connected to AdventureWorksDW2025
✓ Database connection closed

📊 Configured Models:
  - Claude Sonnet 4.5 (anthropic)
    Model ID: claude-sonnet-4-5-20250929
    Cost: $3.0/M input, $15.0/M output

  - GPT 4.1 (openai)
    Model ID: gpt-4-turbo
    Cost: $10.0/M input, $30.0/M output

  - GPT 4o Mini (openai)
    Model ID: gpt-4o-mini
    Cost: $0.15/M input, $0.6/M output



## 3. View Test Cases

In [28]:
# Select first 5 test cases for quick test
test_case_ids = [1, 2, 3, 4, 5]
selected_cases = [tc for tc in TEST_CASES if tc['id'] in test_case_ids]

print(f"\n🎯 Test Cases ({len(selected_cases)} cases):")
print("="*80)
for case in selected_cases:
    print(f"\n{case['id']}. [{case['category']}] - {case['difficulty']}")
    print(f"   Query: {case['natural_language']}")
    print(f"   Expected Tables: {', '.join(case['expected_tables'])}")


🎯 Test Cases (5 cases):

1. [Simple Select] - Easy
   Query: Show me all products
   Expected Tables: DimProduct

2. [Simple Select] - Easy
   Query: List all customers with their email addresses
   Expected Tables: DimCustomer

3. [Simple Filter] - Easy
   Query: Show me products that are red in color
   Expected Tables: DimProduct

4. [Aggregation] - Medium
   Query: What is the total sales amount?
   Expected Tables: FactInternetSales

5. [Aggregation] - Easy
   Query: How many customers do we have?
   Expected Tables: DimCustomer


## 4. Run Quick Evaluation

This will test all 3 models on the 5 test cases above (15 total queries).

In [29]:
# Run the evaluation
print("\n🚀 Starting Quick Evaluation...")
print(f"Testing {len(evaluator.models)} models on {len(test_case_ids)} cases")
print(f"Total queries: {len(evaluator.models) * len(test_case_ids)}")
print("="*80)

results = evaluator.run_full_evaluation(test_case_ids=test_case_ids)

print("\n✅ Evaluation Complete!")


🚀 Starting Quick Evaluation...
Testing 3 models on 5 cases
Total queries: 15

Starting LLM Evaluation
Test Cases: 5
Models: 3


Test Case 1: Show me all products
Category: Simple Select | Difficulty: Easy
  Testing Claude Sonnet 4.5... ✓ (2.23s, $0.015258)
  Testing GPT 4.1... ✓ (0.80s, $0.040390)
  Testing GPT 4o Mini... ✓ (0.90s, $0.000608)

Test Case 2: List all customers with their email addresses
Category: Simple Select | Difficulty: Easy
  Testing Claude Sonnet 4.5... ✓ (2.21s, $0.015912)
  Testing GPT 4.1... ✓ (0.91s, $0.040570)
  Testing GPT 4o Mini... ✓ (0.90s, $0.000615)

Test Case 3: Show me products that are red in color
Category: Simple Filter | Difficulty: Easy
  Testing Claude Sonnet 4.5... ✓ (2.08s, $0.016125)
  Testing GPT 4.1... ✓ (1.75s, $0.040610)
  Testing GPT 4o Mini... ✓ (0.79s, $0.000612)

Test Case 4: What is the total sales amount?
Category: Aggregation | Difficulty: Medium
  Testing Claude Sonnet 4.5... ✓ (2.08s, $0.016047)
  Testing GPT 4.1... ✓ (1.69s, $0.

## 5. View Results Summary

In [30]:
import pandas as pd

# Check if results exist
if results is None:
    print("❌ Error: No results available!")
    print("The evaluation may have failed. Please check the previous cell for errors.")
    print("Make sure to run the evaluation cell (section 4) first.")
else:
    # Calculate summary statistics
    summary_data = []
    for model_key, model_results in results['model_results'].items():
        model_name = evaluator.models[model_key]['name']
        
        total = len(model_results)
        successful = sum(1 for r in model_results if r['success'])
        success_rate = (successful / total * 100) if total > 0 else 0
        
        avg_latency = sum(r['latency'] for r in model_results) / total if total > 0 else 0
        total_cost = sum(r['cost'] for r in model_results)
        
        summary_data.append({
            'Model': model_name,
            'Success Rate': f"{success_rate:.1f}%",
            'Successful': f"{successful}/{total}",
            'Avg Latency': f"{avg_latency:.2f}s",
            'Total Cost': f"${total_cost:.4f}"
        })

    df = pd.DataFrame(summary_data)
    print("\n📊 Results Summary:")
    print(df.to_string(index=False))

    print(f"\n💰 Total Cost: ${results['summary']['total_cost']:.4f}")
    print(f"⏱️  Total Time: {results['summary']['total_time']:.2f}s")


📊 Results Summary:
            Model Success Rate Successful Avg Latency Total Cost
Claude Sonnet 4.5       100.0%        5/5       2.03s    $0.0787
          GPT 4.1       100.0%        5/5       1.24s    $0.2028
      GPT 4o Mini       100.0%        5/5       1.12s    $0.0031

💰 Total Cost: $0.2846
⏱️  Total Time: 21.94s


## 6. Detailed Results by Test Case

In [31]:
# Show detailed results for each test case
if results is None:
    print("❌ No results to display. Please run the evaluation first (section 4).")
else:
    for case in selected_cases:
        case_id = case['id']
        print(f"\n{'='*80}")
        print(f"Test Case {case_id}: {case['natural_language']}")
        print(f"Category: {case['category']} | Difficulty: {case['difficulty']}")
        print(f"{'='*80}")
        
        for model_key, model_results in results['model_results'].items():
            model_name = evaluator.models[model_key]['name']
            case_result = next((r for r in model_results if r['test_case_id'] == case_id), None)
            
            if case_result:
                status = "✓" if case_result['success'] else "✗"
                print(f"\n{status} {model_name}:")
                print(f"  Generated SQL: {case_result['generated_sql'][:200]}..." if len(case_result['generated_sql']) > 200 else f"  Generated SQL: {case_result['generated_sql']}")
                print(f"  Latency: {case_result['latency']:.2f}s | Cost: ${case_result['cost']:.5f}")
                
                if not case_result['success']:
                    print(f"  Error: {case_result.get('error_message', 'Unknown error')}")


Test Case 1: Show me all products
Category: Simple Select | Difficulty: Easy

✓ Claude Sonnet 4.5:
  Generated SQL: SELECT *
FROM dbo.DimProduct
  Latency: 2.23s | Cost: $0.01526

✓ GPT 4.1:
  Generated SQL: SELECT *
FROM dbo.DimProduct;
  Latency: 0.80s | Cost: $0.04039

✓ GPT 4o Mini:
  Generated SQL: SELECT *
FROM dbo.DimProduct;
  Latency: 0.90s | Cost: $0.00061

Test Case 2: List all customers with their email addresses
Category: Simple Select | Difficulty: Easy

✓ Claude Sonnet 4.5:
  Generated SQL: SELECT 
    CustomerKey,
    CustomerAlternateKey,
    FirstName,
    MiddleName,
    LastName,
    EmailAddress
FROM dbo.DimCustomer
WHERE EmailAddress IS NOT NULL;
  Latency: 2.21s | Cost: $0.01591

✓ GPT 4.1:
  Generated SQL: SELECT CustomerKey, EmailAddress
FROM dbo.DimCustomer;
  Latency: 0.91s | Cost: $0.04057

✓ GPT 4o Mini:
  Generated SQL: SELECT 
    CustomerKey, 
    EmailAddress 
FROM 
    dbo.DimCustomer;
  Latency: 0.90s | Cost: $0.00061

Test Case 3: Show me products t

## 7. Export Results

In [32]:
# Save results to file
if results is None:
    print("❌ No results to export. Please run the evaluation first (section 4).")
else:
    results_dir = parent_dir / 'results'
    results_dir.mkdir(exist_ok=True)

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_file = results_dir / f'quick_test_{timestamp}.json'

    with open(output_file, 'w') as f:
        json.dump(results, f, indent=2)

    print(f"\n💾 Results saved to: {output_file}")


💾 Results saved to: c:\Users\Vamsi Chintalapati\Desktop\DSBA 6010\milestone_3_llm_experiments\results\quick_test_20260203_151419.json
